# Attention From Scratch

Companion notebook for the [Attention Mechanism](https://codemia.io/courses/llm_inside_and_out/attention_mechanism) lesson on Codemia.

**What you will build**: scaled dot-product attention and multi-head attention in pure NumPy, then visualize which tokens attend to which on a 6-token toy sequence.

**Karpathy companion**: [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) builds the same operation inside a real language model. This notebook strips it down to the pure tensor math so you can see every multiply. Adapted in part from karpathy/nanoGPT (MIT).

**Time**: ~30-45 minutes

**GPU required**: no

## Setup

In [ ]:
# Only numpy + matplotlib. No torch on purpose: we want to see every operation.
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed seed so the attention patterns you see match the ones discussed below.
np.random.seed(0)

## Concept recap

Self-attention turns a sequence of token vectors `X` (shape `[T, d]`) into a new sequence of the same shape, where each output position is a weighted average of *all* positions. The weights come from comparing a query vector to every key vector via a dot product, scaled by `sqrt(d_k)` to keep variances sane, then softmaxed into a probability distribution. That distribution is what gets multiplied against the values.

The whole operation is three matrix multiplies plus a softmax. Everything else in a transformer layer is support infrastructure.

## Implementation: single-head scaled dot-product attention

In [ ]:
def softmax(x, axis=-1):
    # Subtract the max for numerical stability (otherwise exp() overflows for large logits).
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V are shape [T, d_k]. Think of each row as one token's query/key/value vector.
    d_k = Q.shape[-1]
    # (T, d_k) @ (d_k, T) -> (T, T). Row i column j is the score: how much does token i want token j?
    scores = Q @ K.T
    # Scale by sqrt(d_k). Without this, dot products grow with dimension and push softmax into one-hot land.
    scores = scores / np.sqrt(d_k)
    # Causal / padding masks zero out disallowed positions by adding -inf before softmax.
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    # Softmax along the "keys" axis so each row sums to 1: it's a distribution over which token to look at.
    weights = softmax(scores, axis=-1)
    # Finally mix values according to those weights. Output shape matches Q.
    output = weights @ V
    return output, weights

## Experiment: attention on a 6-token toy sequence

We pick six tokens and hand-design embeddings so the attention pattern is interpretable: `"the"`, `"cat"`, `"sat"`, `"on"`, `"the"`, `"mat"`. We will arrange the embeddings so that content words (cat, sat, mat) cluster together and function words (the, on) cluster separately. Then the softmax scores should show within-cluster attention.

In [ ]:
tokens = ["the", "cat", "sat", "on", "the", "mat"]
T = len(tokens)
d_model = 8

# Hand-crafted embeddings: content words share a direction, function words share another.
# This gives us a predictable attention pattern for inspection.
content_dir = np.array([1, 1, 0, 0, 1, 0, 0, 1], dtype=float)
function_dir = np.array([0, 0, 1, 1, 0, 1, 1, 0], dtype=float)

is_content = np.array([0, 1, 1, 0, 0, 1])  # the, cat, sat, on, the, mat
# Mix the two directions with a small amount of noise so ties don't dominate.
X = np.stack([
    (is_content[i] * content_dir + (1 - is_content[i]) * function_dir)
    + 0.1 * np.random.randn(d_model)
    for i in range(T)
])
print("X shape:", X.shape)
print("X[0] (the):", np.round(X[0], 2))
print("X[1] (cat):", np.round(X[1], 2))

In [ ]:
# Project X into Q, K, V using random linear maps. In a real transformer these are learned.
# With identity-ish projections, attention reduces to cosine similarity of the raw embeddings.
d_k = 8
W_q = np.eye(d_model, d_k) + 0.05 * np.random.randn(d_model, d_k)
W_k = np.eye(d_model, d_k) + 0.05 * np.random.randn(d_model, d_k)
W_v = np.eye(d_model, d_k) + 0.05 * np.random.randn(d_model, d_k)

Q = X @ W_q
K = X @ W_k
V = X @ W_v

out, weights = scaled_dot_product_attention(Q, K, V)
print("attention matrix shape:", weights.shape)
print("each row sums to 1:", np.allclose(weights.sum(axis=-1), 1.0))

In [ ]:
# Visualize the attention weights. Rows = query token, columns = key token.
# Bright squares mean "the query on this row wants to read from the key on this column."
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(weights, cmap='viridis')
ax.set_xticks(range(T), tokens)
ax.set_yticks(range(T), tokens)
ax.set_xlabel('key (attending to)')
ax.set_ylabel('query (from)')
ax.set_title('Self-attention weights')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Print which column each query picks as its top attention target.
# If the toy embedding worked, content tokens should point at other content tokens.
for i, tok in enumerate(tokens):
    top = np.argsort(weights[i])[::-1][:2]
    picks = ", ".join(f"{tokens[j]}({weights[i, j]:.2f})" for j in top)
    print(f"{tok:<5} -> {picks}")

## Variation 1: causal mask

Language models can only look backward. We encode that by zeroing out every position above the diagonal *before* the softmax. Notice how the first token is forced to attend only to itself, so its attention row is a one-hot.

In [ ]:
# Lower-triangular boolean mask: position (i, j) is True iff j <= i.
causal_mask = np.tril(np.ones((T, T), dtype=bool))
out_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(weights_causal, cmap='viridis')
ax.set_xticks(range(T), tokens)
ax.set_yticks(range(T), tokens)
ax.set_title('Causal self-attention')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# The first row must be [1, 0, 0, 0, 0, 0] because token 0 has only itself to look at.
print("first row:", np.round(weights_causal[0], 3))

## Implementation: multi-head attention

Multi-head attention is the single-head operation applied `h` times in parallel with different projections, then concatenated. The key insight: instead of one big `d_model`-dim attention, we do `h` smaller `d_model/h`-dim attentions. Each head can specialize on a different kind of relationship (syntactic, semantic, positional).

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, n_heads):
    T, d_model = X.shape
    d_head = d_model // n_heads
    # Project once and reshape into heads instead of looping -- this is exactly how torch does it.
    Q = (X @ W_q).reshape(T, n_heads, d_head).transpose(1, 0, 2)  # [h, T, d_head]
    K = (X @ W_k).reshape(T, n_heads, d_head).transpose(1, 0, 2)
    V = (X @ W_v).reshape(T, n_heads, d_head).transpose(1, 0, 2)
    head_outs = []
    head_weights = []
    # Run each head's attention independently. In production this is one batched matmul.
    for h in range(n_heads):
        out_h, w_h = scaled_dot_product_attention(Q[h], K[h], V[h])
        head_outs.append(out_h)
        head_weights.append(w_h)
    # Concatenate heads back to [T, d_model] and apply the output projection.
    concat = np.concatenate(head_outs, axis=-1)
    return concat @ W_o, np.stack(head_weights)

In [ ]:
n_heads = 4
# Re-initialize projections so each head gets a different random slice of the space.
W_q_mh = np.random.randn(d_model, d_model) * 0.3
W_k_mh = np.random.randn(d_model, d_model) * 0.3
W_v_mh = np.random.randn(d_model, d_model) * 0.3
W_o_mh = np.random.randn(d_model, d_model) * 0.3

out_mh, weights_mh = multi_head_attention(X, W_q_mh, W_k_mh, W_v_mh, W_o_mh, n_heads)
print("output shape:", out_mh.shape)
print("per-head weights shape:", weights_mh.shape)  # [h, T, T]

In [ ]:
# Plot each head's attention pattern. Heads often develop distinct "roles"
# like "look at the previous token" or "look at matching brackets" after training.
fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
for h in range(n_heads):
    ax = axes[h]
    ax.imshow(weights_mh[h], cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'head {h}')
    ax.set_xticks(range(T), tokens, rotation=45)
    ax.set_yticks(range(T), tokens)
plt.tight_layout()
plt.show()

## Variation 2: what happens without the sqrt(d_k) scaling?

Remove the scale and look at the softmax distribution. With large `d_k`, raw dot products have variance `d_k`, which pushes softmax toward one-hot and kills gradients. Try it and watch the entropy collapse.

In [ ]:
def attention_no_scale(Q, K, V):
    scores = Q @ K.T  # no /sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# Crank d_k up so the effect is dramatic.
d_big = 256
Q_big = np.random.randn(T, d_big)
K_big = np.random.randn(T, d_big)
V_big = np.random.randn(T, d_big)

_, w_scaled = scaled_dot_product_attention(Q_big, K_big, V_big)
_, w_unscaled = attention_no_scale(Q_big, K_big, V_big)

# Entropy in nats. Lower entropy = sharper distribution = gradient vanishes through softmax.
def entropy(w):
    return -(w * np.log(w + 1e-12)).sum(-1).mean()

print(f"entropy WITH  scale: {entropy(w_scaled):.3f} nats (max = {np.log(T):.3f})")
print(f"entropy WITHOUT scale: {entropy(w_unscaled):.3f} nats")
print("The unscaled attention collapses to nearly one-hot -- gradients die here.")

## What to try next

- Add a learned bias to the attention scores (ALiBi) and see how it encodes relative position without any embedding.
- Replace softmax with `relu(scores)^2 / sum(...)` (ReLU^2 attention) and compare entropy and training dynamics.
- Port this to PyTorch and verify that `torch.nn.functional.scaled_dot_product_attention` produces identical outputs (up to float precision).
- Plot per-head entropy across 12 random inits and see which heads end up "sharp" vs "diffuse" by chance alone.